# 12 (Kaggle) — Hybrid CNN + ViT trên MoE-D

Bản Kaggle của `12_hybrid_colab`: **không dùng Google Drive**, mọi run ghi vào `outputs/`, cuối notebook **nén thành 1 file
zip** để tải về. Chạy `experiments/stage10_hybrid.yaml` — **2 ViT × 2 CNN = 4 run, fold 0**, recipe mạnh nhất **M0**
(MoE hướng D · `d_model 256` · overlay `r5_nobce` · clin + derm + metadata · 384 px · 20 epoch, patience 5 · batch 8 × 8 = 64).
Mốc so sánh: **M0 fold 0** (Swin-B-384 MoE-D) val Dice **0.601**, AUC 0.933.

| # | Run (`<vit>__abl_<cnn>__f0`) | Nhánh ViT | Nhánh CNN |
|---|---|---|---|
| 1 | `swin_base_384__abl_HC__f0` | Swin-B/384 (IN-22k→1k) | ConvNeXt-B/384 (IN-22k→1k) |
| 2 | `swin_base_384__abl_HE__f0` | Swin-B/384 | EfficientNetV2-M (IN-21k→1k) |
| 3 | `vit_base_384__abl_HC__f0` | ViT-B/16-384 (IN-21k→1k) | ConvNeXt-B/384 |
| 4 | `vit_base_384__abl_HE__f0` | ViT-B/16-384 | EfficientNetV2-M |

### Kiến trúc
Mỗi ảnh đi qua **cả hai** backbone (mỗi backbone dùng chung cho 2 ảnh); mỗi backbone cho **một token riêng** → 6 token vào
gate của MoE-D:
```
clin ─┬─ ViT ─ token clin       ┐
      └─ CNN ─ token clin_cnn   │
derm ─┬─ ViT ─ token derm       ├─ expert theo nguồn + gate (6) ─ 3 head long-tail (τ 0 / 0.5 / 1) ─ p
      └─ CNN ─ token derm_cnn   │
demo ─ MLP ─ token demo         │
MONET ─ MLP ─ token monet       ┘
```

### Thời gian và VRAM (T4 x2, ước lượng)
fp16, batch 8 × accum 8 + grad checkpointing → ~6–8 GB/run, vừa T4 16 GB. Mỗi GPU chạy 1 run.
Ước lượng ~3.5–4 giờ/run với ConvNeXt, ~3 giờ với EfficientNetV2 → 4 run trên 2 GPU **~7 giờ** (< giới hạn 12 giờ).
`TIME_BUDGET_H` chặn không bắt đầu run mới nếu không kịp xong; run chưa chạy thì chạy ở phiên sau với `RESTORE`.

### Cách chạy trên Kaggle
1. **Settings → Accelerator: GPU T4 x2**, **Internet: On**.
2. **Save Version → Save & Run All (Commit)** — chạy nền tới 12 giờ, không cần mở trình duyệt; xong thì tải zip ở tab *Output*.
3. Phiên sau (nếu còn run chưa xong): *Add Input → Your Work → Notebooks* chọn version trước, điền đường dẫn `outputs` của nó
   vào `RESTORE` → run đã xong được chép lại và bỏ qua.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu (tải công khai bằng gdown)
DATA_SOURCE = None    # hoặc đường dẫn Kaggle Dataset, vd "/kaggle/input/milk10k" (khi đó không tải từ link)

# Chạy tiếp: gắn output của version trước làm Input rồi liệt kê thư mục outputs của nó,
# vd "/kaggle/input/<ten-notebook>/fussion/outputs" (hoặc thư mục outputs trong zip đã giải nén làm Dataset)
RESTORE = []
TIME_BUDGET_H = 11.0

# ===================== CHỌN THÍ NGHIỆM =====================
EXPERIMENT = "experiments/stage10_hybrid.yaml"
BACKBONES = ["swin_base_384", "vit_base_384"]   # nhánh ViT
VARIANTS = ["HC", "HE"]                         # nhánh CNN: HC = ConvNeXt-B · HE = EfficientNetV2-M
FOLDS = [0]                                     # cấu hình thắng chạy 5 fold: [0, 1, 2, 3, 4]

In [ ]:
# ===================== THAM SỐ TRAIN (recipe M0) =====================
# Ưu tiên cao nhất: ghi đè mọi config. T4 không có bf16 -> fp16 (như run M0 trên Kaggle, test 0.512).
PARAMS = {
    "train.batch_size": 8,              # T4 16GB; CUDA out of memory -> 4 và accum_steps 16
    "train.accum_steps": 8,             # batch hiệu dụng 64
    "model.grad_checkpointing": True,
    "train.amp_dtype": "fp16",
    "train.epochs": 20,
    "train.patience": 5,
    "seed": 42,
    "predict.tta": True,
}
RUN_SUFFIX = ""

## 1. Cài đặt

In [ ]:
import os, glob
from IPython.display import Markdown, display

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/tmp/MILK10k"               # ngoài /kaggle/working -> dữ liệu không bị lưu vào output notebook
os.environ["MILK10K_ROOT"] = DATA_DIR   # code tự đọc biến này thay cho data.root trong config

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gdown
!git log -1 --format="Commit: %h %s"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"

## 2. Dữ liệu

In [ ]:
src = f"--source {DATA_SOURCE}" if DATA_SOURCE else f'--drive-url "{DRIVE_URL}"'
!python scripts/setup_data.py {src} --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh mô hình hybrid + lưới 4 run (~2 phút, không tải trọng số)
!python -m pytest tests/test_smoke.py -q -p no:cacheprovider -k "hybrid or stage10 or moe"

## 3. Danh sách run và cấu hình thực tế

In [ ]:
restore_arg = ("--restore " + " ".join(RESTORE)) if RESTORE else ""
for r in RESTORE:
    print(r, "->", "OK" if os.path.isdir(r) else "KHÔNG TỒN TẠI (kiểm tra lại đường dẫn Input)")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
select_arg = (f"--backbones {','.join(BACKBONES)} --variants {','.join(VARIANTS)}"
              + (f" --folds {','.join(map(str, FOLDS))}" if FOLDS else ""))
MIN_PER_RUN = {"HC": 225, "HE": 180}            # phút / run trên T4 (ước lượng; số đo thật in ra sau run đầu)
n_runs = len(BACKBONES) * len(VARIANTS) * len(FOLDS)
hours = sum(MIN_PER_RUN[v] for v in VARIANTS) * len(BACKBONES) * len(FOLDS) / 60 / 2
print(f"{n_runs} run, ước tính ~{hours:.1f} giờ trên 2 GPU (giới hạn phiên: {TIME_BUDGET_H} giờ)")
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus auto --est-run-h 3.5 --time-budget-h {TIME_BUDGET_H}

## 5. Kết quả: so với M0 fold 0 + trọng số gate CNN / ViT
Val fold 0 của từng run. `gate CNN` = tổng trọng số gate của 2 token CNN, trung bình trên lesion val (0 = bỏ qua CNN,
1 = chỉ dùng CNN). Mốc M0 fold 0: Dice **0.601**, AUC 0.933. File nộp test của từng run: `outputs/<run>/submission.csv`
(**chưa nộp** — chọn theo val trước; chỉ cấu hình hơn M0 ≥ ~0.02 mới chạy đủ 5 fold rồi nộp 1 lần).

In [ ]:
import json, glob
import pandas as pd
CLASSES = ["AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF", "MAL_OTH", "MEL", "NV", "SCCKA", "VASC"]
M0_F0 = {"Dice val": 0.601, "AUC": 0.933}
rows, per = [], {}
for d in sorted(glob.glob("outputs/*__abl_H[CE]__f*")):
    try:
        m = json.load(open(f"{d}/metrics.json")); oof = pd.read_csv(f"{d}/oof.csv")
    except FileNotFoundError:
        continue
    name, o = d.rstrip("/").split("/")[-1], m["overall"]
    r = {"run": name, "Dice val": o["dice"], "AUC": o["auc"], "Δ Dice vs M0 f0": o["dice"] - M0_F0["Dice val"]}
    g = oof.filter(like="gate_")
    if len(g.columns):
        r["gate CNN"] = g[[c for c in g.columns if c.endswith("_cnn")]].sum(1).mean()
        r.update({c.removeprefix("gate_"): g[c].mean() for c in g.columns})
    rows.append(r)
    y, pred = oof["label"].to_numpy(), oof[CLASSES].to_numpy().argmax(1)
    per[name] = {c: 2 * ((y == k) & (pred == k)).sum() / max((y == k).sum() + (pred == k).sum(), 1)
                 for k, c in enumerate(CLASSES)}
if rows:
    tab = pd.DataFrame(rows).set_index("run")
    display(tab.round(3)); tab.round(4).to_csv("outputs/stage10_summary.csv")
    print("Dice val theo lớp (fold 0):"); display(pd.DataFrame(per).T.round(3))
else:
    print("Chưa có run nào xong trong outputs/")

## Nén kết quả để tải về

In [ ]:
import zipfile, datetime
from pathlib import Path

zip_path = Path("/kaggle/working") / f"12_hybrid_kaggle_outputs_{datetime.datetime.now():%Y%m%d_%H%M}.zip"
src = Path(REPO_DIR) / "outputs"
n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(src.rglob("*")):
        if f.is_file() and f.suffix != ".pt":          # không đóng gói checkpoint
            z.write(f, Path("outputs") / f.relative_to(src))
            n += 1
print(f"Đã nén {n} file -> {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB)")
print("Tải về: tab Output của version notebook (bên phải) -> {} -> Download".format(zip_path.name))